In [1]:
import json, os
import logging
import sys
import glob

In [2]:
SOURCE_DIR   = r"E:\Callproject\add2"
FILE_PATTERN = "**/*_9x.qxp"
RESCAN = True   # Set to True to ignore the cache and re-scan the directory

# CACHE_FILE = os.path.join(SOURCE_DIR, "_file_cache.json")

CACHE_FILE   = r"E:\Callproject\add2\_file_cache.json"
JS_OUT_PATH  = r"E:\Callproject\add2\batch_export.js"
LOG_PATH     = r"E:\Callproject\add2\_batch_export_js.log"
PDF_OUT_DIR  = r"E:\Callproject\add2\pdffolder\pdf_out"   # flat PDF folder

# Create the PDF output directory if it doesn't exist
os.makedirs(PDF_OUT_DIR, exist_ok=True)


In [3]:
def save_file_cache(file_list):
    with open(CACHE_FILE, "w", encoding="utf-8") as fh:
        json.dump(file_list, fh, indent=2)
    log.info("File list cached → %s  (%d files)", CACHE_FILE, len(file_list))

def load_file_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as fh:
            file_list = json.load(fh)
        log.info("Loaded file list from cache  (%d files)", len(file_list))
        return file_list
    return None

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(os.path.join(SOURCE_DIR, "_batch_export.log"), encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)


In [5]:
qxp_files = None if RESCAN else load_file_cache()

if qxp_files is None:
    log.info("Scanning %s ...", SOURCE_DIR)
    pattern = os.path.join(SOURCE_DIR, FILE_PATTERN)
    qxp_files = sorted(glob.glob(pattern, recursive=True))
    save_file_cache(qxp_files)

if not qxp_files:
    log.error("No *_9x.qxp files found under %s", SOURCE_DIR)
    sys.exit(1)

log.info("Found %d QXP file(s) to process.", len(qxp_files))

with open(CACHE_FILE, "r", encoding="utf-8") as f:
    qxp_files = json.load(f)

2026-04-27 15:12:34,350  INFO      Scanning E:\Callproject\add2 ...
2026-04-27 15:12:51,342  INFO      File list cached → E:\Callproject\add2\_file_cache.json  (4915 files)
2026-04-27 15:12:51,342  INFO      Found 4915 QXP file(s) to process.


In [6]:
def js_str(path):
    return path.replace("\\", "\\\\")

# Build the file list as a JS array of objects, one per file
# Each entry carries its own serial number and pre-computed PDF path
entries = []
for i, qxp_path in enumerate(qxp_files):
    serial   = f"{i+1:06d}"                          # e.g. "000001"
    pdf_path = os.path.join(PDF_OUT_DIR, f"{serial}.pdf")
    stem     = os.path.splitext(qxp_path)[0]
    json_path= stem + ".json"
    entries.append({
        "serial":    serial,
        "qxp_path":  qxp_path,
        "pdf_path":  pdf_path,
        "json_path": json_path,
    })

# Also write a master index so you can always reconstruct the mapping
index_path = os.path.join(PDF_OUT_DIR, "_index.json")
with open(index_path, "w", encoding="utf-8") as f:
    json.dump(entries, f, indent=2)
print(f"Master index written → {index_path}")

# Serialize the file list for embedding in JS
def js_obj(e):
    return (
        f'{{'
        f'"serial":"{e["serial"]}",'
        f'"qxpPath":"{js_str(e["qxp_path"])}",'
        f'"pdfPath":"{js_str(e["pdf_path"])}",'
        f'"jsonPath":"{js_str(e["json_path"])}"'
        f'}}'
    )

file_array = ",\n    ".join(js_obj(e) for e in entries)

js = f"""
// batch_export.js  —  auto-generated by Python
// Files : {len(entries)}
// PDFs  → {js_str(PDF_OUT_DIR)}

var OPEN_FLAGS  = 6;      // SUPPRESSWARNINGS(2) | SKIPMISSINGFONT(4)
var CLOSE_FLAGS = 1;      // CLOSE_PROJECT
var PDF_FLAGS   = 65535;  // kOutputUI_SuppressAll

function extractText(dom) {{
    var result = {{
        layout_name: dom.getAttribute("layout-name"),
        boxes: []
    }};
    var boxes = dom.querySelectorAll("qx-box[box-content-type='text']");
    for (var i = 0; i < boxes.length; i++) {{
        var box = boxes[i];
        var boxData = {{
            box_id: box.getAttribute("box-id"),
            paragraphs: []
        }};
        var paras = box.querySelectorAll("qx-p");
        for (var j = 0; j < paras.length; j++) {{
            var text = paras[j].textContent.trim();
            if (text.length > 0) {{
                boxData.paragraphs.push(text);
            }}
        }}
        if (boxData.paragraphs.length > 0) {{
            result.boxes.push(boxData);
        }}
    }}
    return result;
}}

var files = [
    {file_array}
];

var succeeded = 0;
var failed    = 0;

for (var i = 0; i < files.length; i++) {{
    var f = files[i];

    try {{
        var proj   = app.openProject(f.qxpPath, OPEN_FLAGS);
        var layout = proj.getLayoutByIndex(0);

        // ── PDF (best-effort, failures do not affect JSON) ────────────────
        try {{
            layout.exportLayoutAsPDF(f.pdfPath, PDF_FLAGS);
        }} catch(pdfErr) {{
            fs.writeFileSync(f.pdfPath + "_ERROR.txt", f.serial + " | " + f.qxpPath + "\\n" + pdfErr.toString());
        }}

        // ── JSON (this is what matters) ───────────────────────────────────
        var dom      = app.activeLayoutDOM();
        var textData = extractText(dom);
        textData.serial      = f.serial;
        textData.source_file = f.qxpPath;
        textData.pdf_file    = f.pdfPath;
        fs.writeFileSync(f.jsonPath, JSON.stringify(textData, null, 2));

        proj.closeProject(CLOSE_FLAGS);
        succeeded++;

    }} catch (e) {{
        fs.writeFileSync(f.jsonPath + "_ERROR.txt", f.serial + " | " + f.qxpPath + "\\n" + e.toString());
        failed++;
        try {{ app.activeProject().closeProject(CLOSE_FLAGS); }} catch(e2) {{}}
    }}
}}

var summary = {{
    total:     files.length,
    succeeded: succeeded,
    failed:    failed,
    timestamp: new Date().toISOString()
}};
fs.writeFileSync("{js_str(LOG_PATH)}", JSON.stringify(summary, null, 2));
"""

with open(JS_OUT_PATH, "w", encoding="utf-8") as f:
    f.write(js)

print(f"JS script written  → {JS_OUT_PATH}")
print(f"Covers {len(entries)} files.")

Master index written → E:\Callproject\add2\pdffolder\pdf_out\_index.json
JS script written  → E:\Callproject\add2\batch_export.js
Covers 4915 files.
